# 05C - Feature Engineering

Create model-ready features while preserving reproducibility.

In [1]:

import pandas as pd
import numpy as np
df=pd.read_csv("../data/datasets/american_bankruptcy.csv")


## Encode Target

In [2]:

df['target']=df['status_label'].map({'alive':0,'failed':1})


## Identify Numeric Features

In [3]:

numeric=[c for c in df.select_dtypes(include='number').columns if c!='target']
print(numeric)


['year', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18']


## Log Transform Highly Skewed Features

In [4]:

engineered=df.copy()
created=[]
for c in numeric:
    if (engineered[c]>=0).all():
        s=engineered[c].skew()
        if abs(s)>1:
            new=f'{c}_log1p'
            engineered[new]=np.log1p(engineered[c])
            created.append(new)
print("Created:",created)
display(engineered[created].head() if created else "No log features created")


Created: ['X3_log1p', 'X5_log1p', 'X8_log1p', 'X10_log1p', 'X14_log1p', 'X17_log1p']


,X3_log1p,X5_log1p,X8_log1p,X10_log1p,X14_log1p,X17_log1p
0,2.963880,5.820136,5.923592,6.609347,5.104830,5.997653
1,2.974355,5.773277,5.935206,6.555149,4.839388,5.893416
2,3.156830,5.661529,5.901520,6.566952,5.020348,5.993872
3,3.338329,5.564344,4.972099,6.533238,5.320935,5.972875
4,3.320710,5.514416,5.736273,6.565676,4.884777,6.012756


## Year-Based Features

In [5]:

if 'year' in engineered.columns:
    max_year=engineered['year'].max()
    engineered['record_age']=max_year-engineered['year']
display(engineered[['year','record_age']].head() if 'record_age' in engineered.columns else engineered.head())


,year,record_age
0,1999,19
1,2000,18
2,2001,17
3,2002,16
4,2003,15


## Interaction Features

In [6]:

if {'X1','X2'}.issubset(engineered.columns):
    engineered['X1_X2_ratio']=engineered['X1']/(engineered['X2'].replace(0,np.nan))
if {'X3','X4'}.issubset(engineered.columns):
    engineered['X3_X4_sum']=engineered['X3']+engineered['X4']
new_cols=[c for c in engineered.columns if c.endswith('_ratio') or c.endswith('_sum')]
display(engineered[new_cols].head() if new_cols else "No interaction features created")


,X1_X2_ratio,X3_X4_sum
0,0.613687,107.404
1,0.680651,82.944
2,0.829392,49.703
3,0.798016,57.917
4,0.825917,74.171


## Feature Summary

In [7]:

added=[c for c in engineered.columns if c not in df.columns]
summary=pd.DataFrame({'Engineered Features':added})
display(summary)
print("Original shape:",df.shape)
print("Engineered shape:",engineered.shape)


,Engineered Features
0,X3_log1p
1,X5_log1p
2,X8_log1p
3,X10_log1p
4,X14_log1p
5,X17_log1p
6,record_age
7,X1_X2_ratio
8,X3_X4_sum


Original shape: (78682, 22)
Engineered shape: (78682, 31)


## Save Engineered Dataset

In [8]:

engineered.to_csv("engineered_dataset.csv",index=False)
print("Saved: engineered_dataset.csv")


Saved: engineered_dataset.csv


## Recommendations

In [9]:

for t in [
"Create domain-specific financial ratios when definitions are available.",
"Fit transformations only on training data in production pipelines.",
"Validate engineered features using feature importance and SHAP."
]:
    print("-",t)


- Create domain-specific financial ratios when definitions are available.
- Fit transformations only on training data in production pipelines.
- Validate engineered features using feature importance and SHAP.
